In [ ]:
import pyarrow as pa
import pyarrow.lib as palib
import pandas as pd

ARROW_KEY_ERROR = getattr(pa, "ArrowKeyError", Exception)

def _safe_unregister(ext_name: str):
    try:
        return palib.unregister_extension_type(ext_name)
    except ARROW_KEY_ERROR as err:
        if "No type extension with name" in str(err):
            return None
        raise

def _safe_register(ext_type):
    try:
        return palib.register_extension_type(ext_type)
    except ARROW_KEY_ERROR as err:
        ext_name = getattr(ext_type, "extension_name", None)
        if ext_name and "already defined" in str(err):
            _safe_unregister(ext_name)
            return palib.register_extension_type(ext_type)
        raise

# Reset any previous notebook monkey-patch, then install safe wrappers.
pa.unregister_extension_type = _safe_unregister
pa.register_extension_type = _safe_register

# Notebook re-runs/autoreload can leave Arrow extension state inconsistent.
for ext_name in ("arrow.py_extension_type", "pandas.period", "pandas.interval"):
    _safe_unregister(ext_name)

# # Login using e.g. `huggingface-cli login` to access this dataset
# df = pd.read_parquet("hf://datasets/trentmkelly/US-food-nutrient-data/fitness_nutrition_dataset.parquet")
# df.to_csv("./.db/food_data.csv")


创建数据库


In [15]:
dababase_path = "./.db/finace.db"

In [8]:
import sqlite3
with sqlite3.connect(database=dababase_path) as conn:
    df = pd.read_csv("hf://datasets/aegishield/credit_card_transactions/General Credit Card Transactions (1)..csv")
    df.to_sql("finance", conn)

In [10]:
_SQL = """
SELECT * FROM finance



"""

In [11]:
with sqlite3.connect(database=dababase_path) as conn:
    credit_cards = pd.read_sql(_SQL, conn)
credit_cards

,index,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,0,C10001,40.900749,0.818182,95.40,0.00,95.40,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,1,C10002,3202.467416,0.909091,0.00,0.00,0.00,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,2,C10003,2495.148862,1.000000,773.17,773.17,0.00,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,3,C10004,1666.670542,0.636364,1499.00,1499.00,0.00,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,4,C10005,817.714335,1.000000,16.00,16.00,0.00,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8945,8945,C19186,28.493517,1.000000,291.12,0.00,291.12,0.000000,1.000000,0.000000,0.833333,0.000000,0,6,1000.0,325.594462,48.886365,0.500000,6
8946,8946,C19187,19.183215,1.000000,300.00,0.00,300.00,0.000000,1.000000,0.000000,0.833333,0.000000,0,6,1000.0,275.861322,NaN,0.000000,6
8947,8947,C19188,23.398673,0.833333,144.40,0.00,144.40,0.000000,0.833333,0.000000,0.666667,0.000000,0,5,1000.0,81.270775,82.418369,0.250000,6
8948,8948,C19189,13.457564,0.833333,0.00,0.00,0.00,36.558778,0.000000,0.000000,0.000000,0.166667,2,0,500.0,52.549959,55.755628,0.250000,6


In [16]:
#开始外键
conn = sqlite3.connect(database=dababase_path)
conn.execute("PRAGMA foreign_keys = ON")

In [17]:

if "credit_cards" not in globals():
    with sqlite3.connect(database=dababase_path) as conn:
        credit_cards = pd.read_sql("SELECT * FROM finance", conn)

customers = credit_cards[["CUST_ID", "TENURE"]].copy()
account_summary = credit_cards[["CUST_ID", "BALANCE", "BALANCE_FREQUENCY", "CREDIT_LIMIT"]].copy()
purchase_activity = credit_cards[[
    "CUST_ID",
    "PURCHASES",
    "ONEOFF_PURCHASES",
    "INSTALLMENTS_PURCHASES",
    "PURCHASES_FREQUENCY",
    "ONEOFF_PURCHASES_FREQUENCY",
    "PURCHASES_INSTALLMENTS_FREQUENCY",
    "PURCHASES_TRX",
]].copy()
cash_advance_activity = credit_cards[[
    "CUST_ID",
    "CASH_ADVANCE",
    "CASH_ADVANCE_FREQUENCY",
    "CASH_ADVANCE_TRX",
]].copy()
payment_activity = credit_cards[[
    "CUST_ID",
    "PAYMENTS",
    "MINIMUM_PAYMENTS",
    "PRC_FULL_PAYMENT",
]].copy()

with sqlite3.connect(database=dababase_path) as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executescript(
        """
        DROP TABLE IF EXISTS payment_activity;
        DROP TABLE IF EXISTS cash_advance_activity;
        DROP TABLE IF EXISTS purchase_activity;
        DROP TABLE IF EXISTS account_summary;
        DROP TABLE IF EXISTS customers;

        CREATE TABLE customers (
            cust_id TEXT PRIMARY KEY,
            tenure INTEGER
        );

        CREATE TABLE account_summary (
            cust_id TEXT PRIMARY KEY,
            balance REAL,
            balance_frequency REAL,
            credit_limit REAL,
            FOREIGN KEY (cust_id) REFERENCES customers(cust_id)
        );

        CREATE TABLE purchase_activity (
            cust_id TEXT PRIMARY KEY,
            purchases REAL,
            oneoff_purchases REAL,
            installments_purchases REAL,
            purchases_frequency REAL,
            oneoff_purchases_frequency REAL,
            purchases_installments_frequency REAL,
            purchases_trx INTEGER,
            FOREIGN KEY (cust_id) REFERENCES customers(cust_id)
        );

        CREATE TABLE cash_advance_activity (
            cust_id TEXT PRIMARY KEY,
            cash_advance REAL,
            cash_advance_frequency REAL,
            cash_advance_trx INTEGER,
            FOREIGN KEY (cust_id) REFERENCES customers(cust_id)
        );

        CREATE TABLE payment_activity (
            cust_id TEXT PRIMARY KEY,
            payments REAL,
            minimum_payments REAL,
            prc_full_payment REAL,
            FOREIGN KEY (cust_id) REFERENCES customers(cust_id)
        );
        """
    )

    customers.columns = ["cust_id", "tenure"]
    account_summary.columns = ["cust_id", "balance", "balance_frequency", "credit_limit"]
    purchase_activity.columns = [
        "cust_id",
        "purchases",
        "oneoff_purchases",
        "installments_purchases",
        "purchases_frequency",
        "oneoff_purchases_frequency",
        "purchases_installments_frequency",
        "purchases_trx",
    ]
    cash_advance_activity.columns = [
        "cust_id",
        "cash_advance",
        "cash_advance_frequency",
        "cash_advance_trx",
    ]
    payment_activity.columns = [
        "cust_id",
        "payments",
        "minimum_payments",
        "prc_full_payment",
    ]

    customers.to_sql("customers", conn, if_exists="append", index=False)
    account_summary.to_sql("account_summary", conn, if_exists="append", index=False)
    purchase_activity.to_sql("purchase_activity", conn, if_exists="append", index=False)
    cash_advance_activity.to_sql("cash_advance_activity", conn, if_exists="append", index=False)
    payment_activity.to_sql("payment_activity", conn, if_exists="append", index=False)

{
    "customers": len(customers),
    "account_summary": len(account_summary),
    "purchase_activity": len(purchase_activity),
    "cash_advance_activity": len(cash_advance_activity),
    "payment_activity": len(payment_activity),
}



{'customers': 8950,
 'account_summary': 8950,
 'purchase_activity': 8950,
 'cash_advance_activity': 8950,
 'payment_activity': 8950}

In [ ]:
sql_ = """
EXPLAIN QUERY PLAN
SELECT
    cust_id,
    balance,
    balance_frequency,
    credit_limit
FROM account_summary;
WHERE 
"""

In [19]:
with sqlite3.connect(database=dababase_path) as conn:
    result = pd.read_sql(sql_, conn)
result


,id,parent,notused,detail
0,2,0,0,SCAN account_summary


In [ ]:
update_sql = """
DROP TABLE IF EXISTS payment_activity_demo;
CREATE TABLE payment_activity_demo AS
SELECT
    cust_id,
    payments,
    minimum_payments,
    prc_full_payment
FROM payment_activity;

ALTER TABLE payment_activity_demo ADD COLUMN payment_ratio REAL;
ALTER TABLE payment_activity_demo ADD COLUMN payment_segment TEXT;

UPDATE payment_activity_demo
SET
    payment_ratio = CASE
        WHEN minimum_payments IS NULL OR minimum_payments = 0 THEN NULL
        ELSE ROUND(payments / minimum_payments, 2)
    END,
    payment_segment = CASE
        WHEN prc_full_payment >= 0.95 THEN 'full_payment'
        WHEN minimum_payments IS NULL OR minimum_payments = 0 THEN 'no_minimum_due'
        WHEN payments >= minimum_payments THEN 'paid_at_least_minimum'
        ELSE 'below_minimum'
    END;
"""

with sqlite3.connect(database=dababase_path) as conn:
    conn.executescript(update_sql)
    update_examples = pd.read_sql(
        """
        SELECT
            payment_segment,
            COUNT(*) AS customer_count,
            ROUND(AVG(payment_ratio), 2) AS avg_payment_ratio
        FROM payment_activity_demo
        GROUP BY payment_segment
        ORDER BY customer_count DESC
        """,
        conn,
    )

update_examples



In [ ]:
with sqlite3.connect(database=dababase_path) as conn:
    top_balances = pd.read_sql(
        """
        SELECT
            c.cust_id,
            c.tenure,
            a.balance,
            a.credit_limit,
            p.payments,
            pa.purchases
        FROM customers c
        JOIN account_summary a USING (cust_id)
        JOIN payment_activity p USING (cust_id)
        JOIN purchase_activity pa USING (cust_id)
        WHERE a.balance > 5000
        ORDER BY a.balance DESC
        LIMIT 10
        """,
        conn,
    )

    purchase_band_summary = pd.read_sql(
        """
        SELECT
            CASE
                WHEN pa.purchases >= 5000 THEN 'high_spend'
                WHEN pa.purchases >= 1000 THEN 'mid_spend'
                ELSE 'low_spend'
            END AS purchase_band,
            COUNT(*) AS customer_count,
            ROUND(AVG(pa.purchases), 2) AS avg_purchases,
            ROUND(AVG(pm.payments), 2) AS avg_payments
        FROM purchase_activity pa
        JOIN payment_activity pm USING (cust_id)
        GROUP BY purchase_band
        HAVING COUNT(*) >= 100
        ORDER BY avg_purchases DESC
        """,
        conn,
    )

    above_average_credit = pd.read_sql(
        """
        SELECT
            a.cust_id,
            a.balance,
            a.credit_limit,
            p.payments
        FROM account_summary a
        JOIN payment_activity p USING (cust_id)
        WHERE a.credit_limit > (
            SELECT AVG(credit_limit) FROM account_summary
        )
          AND p.payments > (
            SELECT AVG(payments) FROM payment_activity
        )
        ORDER BY p.payments DESC
        LIMIT 10
        """,
        conn,
    )

print('1) JOIN + WHERE + ORDER BY + LIMIT')
print(top_balances.to_string(index=False))
print('\n2) GROUP BY + HAVING')
print(purchase_band_summary.to_string(index=False))
print('\n3) SUBQUERY + FILTER')
print(above_average_credit.to_string(index=False))



In [ ]:
with sqlite3.connect(database=dababase_path) as conn:
    conn.executescript(
        """
        DROP INDEX IF EXISTS idx_account_summary_balance;
        DROP INDEX IF EXISTS idx_payment_activity_payments;
        """
    )

    scan_plan = pd.read_sql(
        """
        EXPLAIN QUERY PLAN
        SELECT cust_id, balance
        FROM account_summary
        WHERE balance > 5000
        """,
        conn,
    )
    scan_plan.insert(0, "scenario", "filter_without_index")

    conn.executescript(
        """
        CREATE INDEX IF NOT EXISTS idx_account_summary_balance
        ON account_summary(balance);
        CREATE INDEX IF NOT EXISTS idx_payment_activity_payments
        ON payment_activity(payments);
        """
    )

    indexed_plan = pd.read_sql(
        """
        EXPLAIN QUERY PLAN
        SELECT cust_id, balance
        FROM account_summary
        WHERE balance > 5000
        """,
        conn,
    )
    indexed_plan.insert(0, "scenario", "filter_with_index")

    join_plan = pd.read_sql(
        """
        EXPLAIN QUERY PLAN
        SELECT
            c.cust_id,
            a.balance,
            p.payments
        FROM customers c
        JOIN account_summary a USING (cust_id)
        JOIN payment_activity p USING (cust_id)
        WHERE a.balance > 5000 AND p.payments > 2000
        """,
        conn,
    )
    join_plan.insert(0, "scenario", "join_query")

plan_examples = pd.concat([scan_plan, indexed_plan, join_plan], ignore_index=True)
print(
    "SQLite exposes SCAN/SEARCH and nested-loop joins in EXPLAIN QUERY PLAN. "
    "Hash joins are common in engines like PostgreSQL or SQL Server, but not in SQLite."
)
plan_examples

